# 🧭 RCA Summary — recon `b7e2d4c8-1a35-4f9e-8d20-3c6a9b1e77bf`

**1 findings** across **1 table pair(s)** · source dialect: `snowflake`

| Verdict | Count | Meaning |
| :-- | --: | :-- |
| 🔧 Migration-induced | 1 | Fix in the migration |
| 📊 Genuine data difference | 0 | Route to the data owner |
| 🔍 Needs review | 0 | Investigate further |
| ✅ Benign / expected | 0 | No action |

## 🔺 Top priorities

| Severity | Location | Verdict | Fix / next step |
| :-- | :-- | :-- | :-- |
| 🔴 High (80) | `edge_geo (schema)` | 🔧 | Reconcile the schema change: column rename + type widening + nullability. |

## 📋 Reconciliation overview (per table pair)

| Target table | Schema | ➖ Missing in target | ➕ Extra in target | 🔤 Mismatched columns | Verdicts |
| :-- | :-: | --: | --: | :-- | :-- |
| `edge_geo` | ⚠️ 1 | · | · | 0 | 🔧1 |

## 📈 Match rates (row & column level)

Reconciliation health per table pair. **Row match %** = source rows that exist in target *and* match on all columns.

| Target table | Source rows | Target rows | ➖ Missing | ➕ Extra | Mismatched rows | ✅ Row match % |
| :-- | --: | --: | --: | --: | --: | --: |
| `edge_geo` | 800 | 800 | 0 | 0 | 0 | **100.00%** |


## 🎯 Findings by verdict _(highest impact first)_

## 🔧 Migration-induced — _Fix in the migration_

| Location | Severity | Category | Conf. | ✔ | Root cause |
| :-- | :-- | :-- | :-: | :-: | :-- |
| `edge_geo (schema)` | 🔴 High | 🔢 type_precision | 92% | · | column renamed + type widened (DECIMAL(9,6)->DOUBLE) + nullability change |

---
# 📅 Validation (row & column match %, date-range filterable)

Set `start_date` / `end_date` widgets to validate a slice, then re-run.

In [ ]:
# 📅 Date-range validation — set the window (widgets), then re-run these cells.
# Row match % and per-column match % over an optional date range so you can
# validate a slice of the migration (e.g. one month) rather than the whole table.
dbutils.widgets.text("start_date", "2000-01-01")
dbutils.widgets.text("end_date", "2100-01-01")
START, END = dbutils.widgets.get("start_date"), dbutils.widgets.get("end_date")

def _win(date_col):
    return f"WHERE `{date_col}` BETWEEN '{START}' AND '{END}'" if date_col else ""

def validate_rows(src, tgt, keys, date_col=None):
    name = tgt.split(".")[-1]
    if not keys:  # no join key learned — report counts only (edit keys to enable match)
        return spark.sql(f"""
            SELECT '{name}' AS table,
                   (SELECT count(*) FROM {src} {_win(date_col)}) AS source_rows,
                   (SELECT count(*) FROM {tgt} {_win(date_col)}) AS target_rows,
                   CAST(NULL AS BIGINT) AS matched_keys,
                   CAST(NULL AS DOUBLE) AS row_match_pct
        """)
    on = " AND ".join(f"s.`{k}` = t.`{k}`" for k in keys)
    return spark.sql(f"""
        WITH s AS (SELECT * FROM {src} {_win(date_col)}),
             t AS (SELECT * FROM {tgt} {_win(date_col)})
        SELECT '{name}' AS table,
               (SELECT count(*) FROM s) AS source_rows,
               (SELECT count(*) FROM t) AS target_rows,
               (SELECT count(*) FROM s JOIN t ON {on}) AS matched_keys,
               round(100.0 * (SELECT count(*) FROM s JOIN t ON {on}) /
                     nullif((SELECT count(*) FROM s), 0), 2) AS row_match_pct
    """)

def validate_column(src, tgt, keys, col, date_col=None):
    on = " AND ".join(f"s.`{k}` = t.`{k}`" for k in keys) if keys else "TRUE"
    return spark.sql(f"""
        WITH s AS (SELECT * FROM {src} {_win(date_col)}),
             t AS (SELECT * FROM {tgt} {_win(date_col)})
        SELECT '{col}' AS column, count(*) AS compared,
               sum(CASE WHEN s.`{col}` <=> t.`{col}` THEN 1 ELSE 0 END) AS matches,
               round(100.0 * sum(CASE WHEN s.`{col}` <=> t.`{col}` THEN 1 ELSE 0 END) /
                     nullif(count(*), 0), 2) AS match_pct
        FROM s JOIN t ON {on}
    """)


In [ ]:
# Row-level match per table pair (edit date_col via the widgets above):
row_checks = [
    validate_rows("fevm_ps_dr_us_east_2_catalog.mig_edge_source.edge_geo", "fevm_ps_dr_us_east_2_catalog.mig_edge_target.edge_geo", ['geo_id'], None),
]
from functools import reduce
reduce(lambda a, b: a.unionByName(b), row_checks).display()

In [ ]:
# No column-level mismatches to validate.

---
# 🔬 Findings & evidence

Grouped by table pair (as Lakebridge reports), then schema → row-level → column-level. Each finding shows the concluded verdict and the query that confirms it. Re-run any cell to drill deeper.

## 📦 `fevm_ps_dr_us_east_2_catalog.mig_edge_target.edge_geo`  
_🔧1  ·  1 finding(s)_

### 🔧 `edge_geo (schema)` — Migration-induced

- **Category**: 🔢 type_precision  ·  **Confidence**: 92%  ·  **Owner**: migration engineer
- **Signal**: **schema-level** — 1 column datatype difference(s) reported by recon
- **Root cause**: column renamed + type widened (DECIMAL(9,6)->DOUBLE) + nullability change
- **Fix**: Reconcile the schema change: column rename + type widening + nullability.
- **Inputs used**: 📊 recon data

In [ ]:
# Re-run to confirm / drill deeper for edge_geo (schema)
# spark.sql("SELECT * FROM fevm_ps_dr_us_east_2_catalog.mig_edge_target.edge_geo LIMIT 20").display()

---
# 🧾 Conclusion & recommended actions

Analyzed **1 findings**. Every verdict below is backed by a query executed in this notebook (see the cell under each finding).

## 🔧 Fix in the migration — 1 (owner: migration engineer)
- `edge_geo (schema)` — Reconcile the schema change: column rename + type widening + nullability.

## 📊 Route to the data owner — 0 (not migration bugs)
- _None._

## ✅ Benign / expected — 0
- 0 finding(s) are representation-only or within tolerance; no action.

> If re-running a cell changes an output, update that finding's verdict above and regenerate this report so the conclusion always matches the evidence.